## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page [JSL Information Extraction Pipeline](https://aws.amazon.com/marketplace).
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms.
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

## JSL Information Extraction Pipeline

This pipeline extracts structured text content from medical documents and plain text strings.
It supports a wide range of input formats including PDF, scanned images (OCR), Word documents, PowerPoint, Excel, and plain text.

For each document the pipeline returns:
- **`note_text`**: full extracted plain text
- **`note_json`**: structured extraction including tables, pictures, and sections
- **`output_s3_uri`**: S3 location of the saved result (when `output_s3_folder_path` is provided)

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import base64
import json
import os
import time

import boto3
import pandas as pd
import sagemaker as sage
from sagemaker import ModelPackage, get_execution_role
from urllib.parse import urlparse

In [ ]:
sagemaker_session = sage.Session()
s3_bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
role = get_execution_role()

sm_runtime = boto3.client("sagemaker-runtime")
s3_client = sagemaker_session.boto_session.client("s3")

pd.set_option("display.max_colwidth", None)

In [ ]:
model_name = "information-extraction-pipeline"
real_time_inference_instance_type = "ml.m5.2xlarge"
batch_transform_inference_instance_type = "ml.m5.2xlarge"

## 2. Create a deployable model from the model package.

In [ ]:
model = ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sagemaker_session,
)

### Input Format

The endpoint accepts `application/json`. The request body supports the following fields:

| Field | Type | Description |
|---|---|---|
| `texts` | `list[str]` | List of plain-text strings to process inline. |
| `s3_file_path` | `str` | S3 URI of a single document (PDF, image, Word, etc.). |
| `s3_folder_path` | `str` | S3 URI prefix of a folder of documents. |
| `file_bytes` | `str` | Base64-encoded document bytes. Requires `filename`. |
| `filename` | `str` | Filename for the uploaded `file_bytes` (determines file type). |
| `output_s3_folder_path` | `str` | S3 folder to write results to. If omitted, results are returned inline. |

Provide exactly one of `texts`, `s3_file_path`, `s3_folder_path`, or `file_bytes` per request.

### Output Format

`predictions` is a map of `doc_id` to extraction result. For inline `texts` inputs the keys are `doc_0`, `doc_1`, etc. For file inputs the key is the original filename.

`output_s3_uri` is only present per document when `output_s3_folder_path` was set in the request — it is the URI of the written result file, not a folder.

The `extraction_info.engine` field indicates what extraction method was used: `"text"` (plain text pass-through), `"adaptive"` (auto-selected), `"docling"` (document parsing), or `"granite"` (vision model).

```json
{
  "consumed_units": 1,
  "predictions": {
    "doc_0": {
      "doc_id": "doc_0",
      "note_text": "Patient is a 58-year-old male with stage III lung cancer...",
      "note_json": {
        "tables": [],
        "pictures": [],
        "sections": {},
        "extraction_info": {
          "engine": "text",
          "pages": 1
        }
      }
    }
  }
}
```

For document inputs (PDF, image, Word, etc.), `note_json` may contain structured tables, pictures, and sections extracted from the document.

## 3. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

### A. Deploy the SageMaker model to an endpoint

In [ ]:
predictor = model.deploy(
    initial_instance_count=1,
    instance_type=real_time_inference_instance_type,
    endpoint_name=model_name,
)

Once the endpoint has been created, you can perform real-time inference.

In [ ]:
def invoke_endpoint(payload):
    response = sm_runtime.invoke_endpoint(
        EndpointName=model_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read().decode("utf-8"))

### B. Inference with inline text

In [ ]:
sample_texts = [
    "Patient is a 58-year-old male with stage III lung cancer. EGFR mutation positive. Started on erlotinib 150mg daily.",
    "62-year-old female with Type 2 diabetes mellitus, HbA1c 9.2%. Started metformin 500mg twice daily. Blood pressure 140/85 mmHg.",
]

In [ ]:
result = invoke_endpoint({"texts": sample_texts})

for doc_id, doc in result["predictions"].items():
    print(f"--- {doc_id} ---")
    print(f"Extracted text: {doc['note_text']}")
    print(f"Pages: {doc['note_json']['extraction_info']['pages']}")
    print(f"Tables found: {len(doc['note_json']['tables'])}")
    print(f"Sections found: {len(doc['note_json']['sections'])}")
    print()

### C. Inference from an S3 document

In [ ]:
result = invoke_endpoint({"s3_file_path": f"s3://{s3_bucket}/test-docs/11252162-RR-15.pdf"})
print(json.dumps(result, indent=2))

### D. Inference from an S3 folder

In [ ]:
result = invoke_endpoint({"s3_folder_path": f"s3://{s3_bucket}/test-docs/"})
print(json.dumps(result, indent=2))

### E. Inference from a local file (base64 upload)

In [ ]:
# Download the sample document from S3 to a local temp path
local_file = "/tmp/sample.pdf"
s3_client.download_file(s3_bucket, "test-docs/11252162-RR-15.pdf", local_file)

with open(local_file, "rb") as f:
    file_bytes = f.read()

result = invoke_endpoint({
    "file_bytes": base64.b64encode(file_bytes).decode("utf-8"),
    "filename": os.path.basename(local_file),
    "output_s3_folder_path": f"s3://{s3_bucket}/{model_name}/output/",
})

for doc_id, doc in result["predictions"].items():
    print(f"Result written to: {doc['output_s3_uri']}")
    print(json.dumps(doc, indent=2))

### F. Delete the endpoint

Now that you have successfully performed real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(model_name)
model.sagemaker_session.delete_endpoint_config(model_name)

## 4. Batch inference

In [ ]:
validation_input_path = f"s3://{s3_bucket}/{model_name}/validation-input/json/"
validation_output_path = f"s3://{s3_bucket}/{model_name}/validation-output/json/"

def upload_to_s3(data, file_name):
    s3_client.put_object(
        Bucket=s3_bucket,
        Key=f"{model_name}/validation-input/json/{file_name}",
        Body=data.encode("utf-8"),
    )

In [ ]:
input_data = {
    "input1.json": json.dumps({
        "texts": [
            "Patient is a 58-year-old male with stage III lung cancer. EGFR mutation positive. Started on erlotinib 150mg daily."
        ]
    }),
    "input2.json": json.dumps({
        "texts": [
            "Patient is a 58-year-old male with stage III lung cancer. EGFR mutation positive. Started on erlotinib 150mg daily.",
            "62-year-old female with Type 2 diabetes mellitus, HbA1c 9.2%. Started metformin 500mg twice daily. Blood pressure 140/85 mmHg.",
        ]
    }),
}

for file_name, data in input_data.items():
    upload_to_s3(data, file_name)
    print(f"Uploaded {file_name}")

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type=batch_transform_inference_instance_type,
    accept="application/json",
    output_path=validation_output_path,
)
transformer.transform(validation_input_path, content_type="application/json")
transformer.wait()

In [ ]:
def retrieve_output(file_name):
    parsed_url = urlparse(transformer.output_path)
    file_key = f"{parsed_url.path[1:]}{file_name}.out"
    response = s3_client.get_object(Bucket=s3_bucket, Key=file_key)
    return json.loads(response["Body"].read().decode("utf-8"))

for file_name in input_data:
    print(f"--- {file_name} ---")
    result = retrieve_output(file_name)
    for doc_id, doc in result["predictions"].items():
        print(f"  {doc_id}: {doc['note_text'][:80]}...")
    print()

In [ ]:
model.delete_model()

### Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model.

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__ to cancel the subscription.
